> **Research provenance.** This notebook records the original empirical workflow. Licensed inputs, saved forecasts and private research infrastructure are not distributed. Outputs and attachments are removed; see [notebook configuration](README.md). The maintained public HMM includes post-study correctness hardening and has not been rerun across the complete 2001-2024 historical sample. Saved paper-era HMM results are not replication targets for the maintained implementation.


# Economic Mechanism Audit

This notebook is a **post-processing analysis of saved research outputs**. It does not rerun the CNN, RF, HMM, portfolio optimizer, fixed-AUM capacity grid, or Dynamic NAV.

## Research questions

1. **Sector diversification and risk** — Does HMM's lower volatility coincide with lower sector concentration?
2. **Turnover versus liquidity** — Is HMM's higher cost driven mainly by more trading or by worse liquidity?
3. **Short-sleeve capacity** — How much of the HMM–RF cost/turnover gap originates in the short sleeve?
4. **Smoothing versus regime timing** — Do state variables and changes in CNN/RF influence explain HMM turnover and replacement?
5. **Capacity-induced alpha dilution** — As AUM rises, do portfolios become broader/slower while executed gross Sharpe falls?
6. **Asymmetric hybrid diagnostic** — What do the saved sleeve streams imply for HMM-long/RF-short versus RF-long/HMM-short?

## Claim discipline

These are mechanism diagnostics, not causal identification. The asymmetric hybrid is not a jointly reoptimized strategy.

In [ ]:
import os
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from IPython.display import display


# =============================================================================
# Configuration
# =============================================================================

PERIODS_PER_YEAR = 52
MODELS = ("RF", "HMM")
PRIMARY_AUM = 100_000_000.0
HAC_LAGS = 8


# =============================================================================
# Saved research outputs
# =============================================================================

if not os.environ.get("FUSION_ANALYSIS_DIR"):
    raise RuntimeError("Set FUSION_ANALYSIS_DIR to the existing saved analysis directory.")
ANALYSIS_OUTPUT_DIR = Path(os.environ["FUSION_ANALYSIS_DIR"]).expanduser().resolve()

CAPACITY_DIR = (
    ANALYSIS_OUTPUT_DIR
    / "fixed_aum_capacity"
)

AUDIT_DIR = (
    ANALYSIS_OUTPUT_DIR
    / "economic_mechanism_audit"
)

FIGURE_DIR = (
    AUDIT_DIR
    / "figures"
)


# =============================================================================
# Validation
# =============================================================================

if not ANALYSIS_OUTPUT_DIR.exists():
    raise FileNotFoundError(
        f"research output directory does not exist:\n"
        f"{ANALYSIS_OUTPUT_DIR}"
    )

if not CAPACITY_DIR.exists():
    raise FileNotFoundError(
        f"Fixed-AUM capacity directory does not exist:\n"
        f"{CAPACITY_DIR}"
    )

capacity_summary_path = (
    CAPACITY_DIR
    / "fixed_aum_capacity_summary.csv"
)

if not capacity_summary_path.exists():
    raise FileNotFoundError(
        f"Capacity summary does not exist:\n"
        f"{capacity_summary_path}"
    )


# =============================================================================
# New audit output directory
# =============================================================================

AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


print("research output:")
print(ANALYSIS_OUTPUT_DIR)

print("\nFixed-AUM capacity:")
print(CAPACITY_DIR)

print("\nEconomic-mechanism audit:")
print(AUDIT_DIR)

print("\nSetup: PASS")

## 1. Load saved fixed-AUM run details

Expected saved files per model/AUM are `weekly_accounting.csv`, `portfolio_returns.csv`,
`tc_diagnostics.csv`, `portfolio_diagnostics.csv`, and `summary.json`.

In [ ]:
def safe_name(x):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(x)).strip("_.") or "model"

def run_dir(model, aum):
    return CAPACITY_DIR / "runs" / safe_name(model) / f"aum_{int(round(float(aum)))}"

def read_req(path, **kwargs):
    if not path.exists():
        raise FileNotFoundError(f"Missing saved artifact: {path}")
    return pd.read_csv(path, **kwargs)

def norm_dates(df):
    out = df.copy()
    for c in ("date", "Date"):
        if c in out.columns:
            out[c] = pd.to_datetime(out[c], errors="raise").dt.normalize()
    return out

capacity_summary = read_req(CAPACITY_DIR / "fixed_aum_capacity_summary.csv")
capacity_summary["aum_dollars"] = pd.to_numeric(capacity_summary["aum_dollars"], errors="raise")
sel = capacity_summary[capacity_summary["model"].astype(str).isin(MODELS)].copy()

pf_frames, tc_frames, acct_frames = [], [], []
portfolio_returns = {}

for r in sel.itertuples(index=False):
    model, aum = str(r.model), float(r.aum_dollars)
    d = run_dir(model, aum)

    pf = norm_dates(read_req(d / "portfolio_diagnostics.csv"))
    tc = norm_dates(read_req(d / "tc_diagnostics.csv"))
    ac = norm_dates(read_req(d / "weekly_accounting.csv"))
    pr = read_req(d / "portfolio_returns.csv", index_col=0)
    pr.index = pd.to_datetime(pr.index, errors="raise").normalize()

    for f in (pf, tc, ac):
        f["model"] = model
        f["aum_dollars"] = aum
        f["aum_millions"] = aum / 1e6

    pf_frames.append(pf)
    tc_frames.append(tc)
    acct_frames.append(ac)
    portfolio_returns[(model, aum)] = pr

portfolio_diag = pd.concat(pf_frames, ignore_index=True)
tc_diag = pd.concat(tc_frames, ignore_index=True)
accounting = pd.concat(acct_frames, ignore_index=True)

required = {
    "date","linear_cost","impact_cost","total_cost","low_cost","high_cost",
    "total_turnover","low_turnover","high_turnover","low_effective_n","high_effective_n",
    "low_total_names","high_total_names","low_sector_hhi","high_sector_hhi",
    "low_dom_sector_share","high_dom_sector_share","any_pos_cap_bind","any_trade_cap_bind",
    "pos_cap_bind_weight_share","trade_cap_bind_weight_share",
    "low_avg_spread_bps","high_avg_spread_bps","low_avg_adv_musd","high_avg_adv_musd",
    "low_avg_sigma","high_avg_sigma","low_replacement_frac","high_replacement_frac",
}
missing = sorted(required - set(portfolio_diag.columns))
if missing:
    raise RuntimeError("Required portfolio diagnostic columns missing: " + ", ".join(missing))

print("Saved-run loading: PASS")
print("AUM grid ($m):", sorted(sel["aum_millions"].unique()))

In [ ]:
def ann_stats(x):
    x = pd.to_numeric(pd.Series(x), errors="coerce").dropna()
    mean = x.mean() * PERIODS_PER_YEAR
    vol = x.std(ddof=1) * np.sqrt(PERIODS_PER_YEAR)
    return {"n": len(x), "ann_mean": mean, "ann_vol": vol, "sharpe": mean / vol if vol > 0 else np.nan}

def hac(df, y, xs):
    d = df[[y, *xs]].apply(pd.to_numeric, errors="coerce").dropna()
    X = sm.add_constant(d[xs], has_constant="add")
    fit = sm.OLS(d[y], X).fit(cov_type="HAC", cov_kwds={"maxlags": HAC_LAGS})
    rows = []
    for v in fit.params.index:
        rows.append({
            "dependent": y, "variable": v, "n": int(fit.nobs),
            "beta": float(fit.params[v]), "hac_t": float(fit.tvalues[v]),
            "p_value_two_sided": float(fit.pvalues[v]), "r_squared": float(fit.rsquared),
        })
    return pd.DataFrame(rows)

def pair(df, cols, aum=None):
    d = df.copy()
    if aum is not None:
        d = d[np.isclose(d["aum_dollars"], float(aum))]
    keys = ["date", "aum_dollars", "aum_millions"]
    pieces = []
    for c in cols:
        p = d.pivot_table(index=keys, columns="model", values=c, aggfunc="last")
        p.columns = [f"{c}_{m}" for m in p.columns]
        pieces.append(p)
    return pd.concat(pieces, axis=1).reset_index()

def savefig(fig, name):
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / name, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)

# Question 1 — Sector diversification and lower HMM volatility

The diagnostic asks whether HMM has lower sleeve-level sector HHI and whether the **RF − HMM HHI gap**
is associated with the **RF − HMM realized-volatility gap**.

In [ ]:
sector = portfolio_diag.copy()
sector["mean_sector_hhi"] = (sector["low_sector_hhi"] + sector["high_sector_hhi"]) / 2
sector["mean_dom_sector_share"] = (sector["low_dom_sector_share"] + sector["high_dom_sector_share"]) / 2
sector["mean_effective_n"] = (sector["low_effective_n"] + sector["high_effective_n"]) / 2

sec_aum = sector.groupby(["model","aum_dollars","aum_millions"], as_index=False).agg(
    mean_sector_hhi=("mean_sector_hhi","mean"),
    mean_dom_sector_share=("mean_dom_sector_share","mean"),
    mean_effective_n=("mean_effective_n","mean"),
)

risk = []
for (m,a), g in accounting.groupby(["model","aum_dollars"]):
    s = ann_stats(g["executed_gross_return"])
    risk.append({"model":m,"aum_dollars":a,"executed_gross_ann_vol":s["ann_vol"],"executed_gross_sharpe":s["sharpe"]})
risk = pd.DataFrame(risk)

sec_aum = sec_aum.merge(risk, on=["model","aum_dollars"], validate="one_to_one")
display(sec_aum.sort_values(["aum_dollars","model"]).style.format({
    "aum_millions":"${:,.0f}m","mean_sector_hhi":"{:.3f}",
    "mean_dom_sector_share":"{:.1%}","mean_effective_n":"{:.1f}",
    "executed_gross_ann_vol":"{:.2%}","executed_gross_sharpe":"{:.3f}",
}))
sec_aum.to_csv(AUDIT_DIR / "sector_diversification_by_aum.csv", index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
for m,g in sec_aum.groupby("model"):
    g=g.sort_values("aum_millions")
    ax.plot(g["aum_millions"],g["mean_sector_hhi"],marker="o",label=m)
ax.set_xscale("log")
ax.set_xlabel("AUM ($m, log scale)")
ax.set_ylabel("Mean sleeve sector HHI")
ax.set_title("Sector Concentration by AUM")
ax.legend()
savefig(fig,"sector_hhi_by_aum.png")

In [ ]:
p_hhi = pair(sector[["date","model","aum_dollars","aum_millions","mean_sector_hhi"]], ["mean_sector_hhi"], PRIMARY_AUM)
p_ret = pair(accounting[["date","model","aum_dollars","aum_millions","executed_gross_return"]], ["executed_gross_return"], PRIMARY_AUM)
sr = p_hhi.merge(p_ret, on=["date","aum_dollars","aum_millions"], validate="one_to_one").sort_values("date")
sr["rf_minus_hmm_hhi"] = sr["mean_sector_hhi_RF"] - sr["mean_sector_hhi_HMM"]
sr["rf_minus_hmm_sq_return"] = sr["executed_gross_return_RF"]**2 - sr["executed_gross_return_HMM"]**2
sr["rf_26w_vol"] = sr["executed_gross_return_RF"].rolling(26,min_periods=13).std()*np.sqrt(PERIODS_PER_YEAR)
sr["hmm_26w_vol"] = sr["executed_gross_return_HMM"].rolling(26,min_periods=13).std()*np.sqrt(PERIODS_PER_YEAR)
sr["rf_minus_hmm_26w_vol"] = sr["rf_26w_vol"] - sr["hmm_26w_vol"]
sr["rf_minus_hmm_hhi_26w"] = sr["rf_minus_hmm_hhi"].rolling(26,min_periods=13).mean()

sector_hac = hac(sr,"rf_minus_hmm_sq_return",["rf_minus_hmm_hhi"])
rolling_corr = sr[["rf_minus_hmm_hhi_26w","rf_minus_hmm_26w_vol"]].dropna().corr().iloc[0,1]

display(sector_hac.style.format({"beta":"{:+.6f}","hac_t":"{:.3f}","p_value_two_sided":"{:.4f}","r_squared":"{:.4f}"}))
print(f"26-week HHI-gap / volatility-gap correlation: {rolling_corr:+.3f}")
sector_hac.to_csv(AUDIT_DIR/"sector_hhi_risk_hac_100m.csv",index=False)

fig,ax=plt.subplots(figsize=(8,5))
d=sr[["rf_minus_hmm_hhi_26w","rf_minus_hmm_26w_vol"]].dropna()
ax.scatter(d["rf_minus_hmm_hhi_26w"],d["rf_minus_hmm_26w_vol"],alpha=.45)
ax.axhline(0,linewidth=1); ax.axvline(0,linewidth=1)
ax.set_xlabel("RF − HMM sector HHI (26-week mean)")
ax.set_ylabel("RF − HMM annualized realized volatility")
ax.set_title("Sector Concentration and Relative Realized Risk — $100m")
savefig(fig,"sector_hhi_vs_relative_volatility_100m.png")

# Question 2 — Turnover versus liquidity in the HMM–RF cost gap

Using \(C=T\times c\), the exact symmetric decomposition is

\[
C_H-C_R
=(T_H-T_R)\frac{c_H+c_R}{2}
+(c_H-c_R)\frac{T_H+T_R}{2}.
\]

The first term is the **turnover quantity effect**; the second is the **cost-intensity effect**.
Spread and nonlinear-impact cost gaps are reported separately.

In [ ]:
cb = portfolio_diag[[
    "date","model","aum_dollars","aum_millions","total_turnover","total_cost",
    "linear_cost","impact_cost","low_avg_spread_bps","high_avg_spread_bps",
    "low_avg_adv_musd","high_avg_adv_musd","low_avg_sigma","high_avg_sigma"
]].copy()
cb["cost_intensity"] = np.where(cb["total_turnover"].abs()>1e-12,cb["total_cost"]/cb["total_turnover"],np.nan)
cb["mean_spread_bps"]=(cb["low_avg_spread_bps"]+cb["high_avg_spread_bps"])/2
cb["mean_adv_musd"]=(cb["low_avg_adv_musd"]+cb["high_avg_adv_musd"])/2
cb["mean_sigma"]=(cb["low_avg_sigma"]+cb["high_avg_sigma"])/2

pc = pair(cb,["total_turnover","total_cost","linear_cost","impact_cost","cost_intensity","mean_spread_bps","mean_adv_musd","mean_sigma"])
pc["cost_gap"]=pc["total_cost_HMM"]-pc["total_cost_RF"]
pc["turnover_effect"]=(pc["total_turnover_HMM"]-pc["total_turnover_RF"])*(pc["cost_intensity_HMM"]+pc["cost_intensity_RF"])/2
pc["intensity_effect"]=(pc["cost_intensity_HMM"]-pc["cost_intensity_RF"])*(pc["total_turnover_HMM"]+pc["total_turnover_RF"])/2
pc["residual"]=pc["cost_gap"]-pc["turnover_effect"]-pc["intensity_effect"]
assert pc["residual"].abs().max()<1e-10

pc["linear_gap"]=pc["linear_cost_HMM"]-pc["linear_cost_RF"]
pc["impact_gap"]=pc["impact_cost_HMM"]-pc["impact_cost_RF"]

cost_aum=pc.groupby("aum_millions",as_index=False).agg(
    total_cost_gap=("cost_gap","mean"),turnover_effect=("turnover_effect","mean"),
    intensity_effect=("intensity_effect","mean"),linear_gap=("linear_gap","mean"),
    impact_gap=("impact_gap","mean"),hmm_turnover=("total_turnover_HMM","mean"),
    rf_turnover=("total_turnover_RF","mean"),hmm_spread=("mean_spread_bps_HMM","mean"),
    rf_spread=("mean_spread_bps_RF","mean"),hmm_adv=("mean_adv_musd_HMM","mean"),
    rf_adv=("mean_adv_musd_RF","mean"),
)
for c in ["total_cost_gap","turnover_effect","intensity_effect","linear_gap","impact_gap"]:
    cost_aum[c+"_bps"]=1e4*cost_aum[c]

display(cost_aum.style.format({
    "aum_millions":"${:,.0f}m","total_cost_gap_bps":"{:+.2f}",
    "turnover_effect_bps":"{:+.2f}","intensity_effect_bps":"{:+.2f}",
    "linear_gap_bps":"{:+.2f}","impact_gap_bps":"{:+.2f}",
    "hmm_turnover":"{:.3f}","rf_turnover":"{:.3f}",
    "hmm_spread":"{:.2f}","rf_spread":"{:.2f}",
    "hmm_adv":"${:,.1f}m","rf_adv":"${:,.1f}m",
}))
cost_aum.to_csv(AUDIT_DIR/"hmm_rf_cost_decomposition.csv",index=False)

In [ ]:
for columns, labels, title, filename in [
    (["turnover_effect_bps","intensity_effect_bps"],["Turnover quantity","Cost intensity"],
     "HMM–RF Cost Gap: More Trading vs More Expensive Trading","cost_gap_turnover_vs_intensity.png"),
    (["linear_gap_bps","impact_gap_bps"],["Linear spread","Nonlinear impact"],
     "HMM–RF Cost Gap: Spread vs Market Impact","cost_gap_linear_vs_impact.png"),
]:
    fig,ax=plt.subplots(figsize=(10,5))
    d=cost_aum.sort_values("aum_millions")
    x=np.arange(len(d)); w=.35
    ax.bar(x-w/2,d[columns[0]],w,label=labels[0])
    ax.bar(x+w/2,d[columns[1]],w,label=labels[1])
    ax.axhline(0,linewidth=1)
    ax.set_xticks(x); ax.set_xticklabels([f"${v:,.0f}m" for v in d["aum_millions"]])
    ax.set_ylabel("HMM − RF cost gap (bps/rebalance)")
    ax.set_title(title); ax.legend()
    savefig(fig,filename)

# Question 3 — Short-sleeve capacity bottleneck

Saved diagnostics identify low/high sleeve cost and turnover separately. Cap-binding flags are portfolio-wide, so the notebook does **not** pretend to assign an individual binding constraint to a sleeve.

In [ ]:
sl = pair(portfolio_diag[[
    "date","model","aum_dollars","aum_millions","low_cost","high_cost","low_turnover","high_turnover",
    "any_trade_cap_bind","trade_cap_bind_weight_share","any_pos_cap_bind","pos_cap_bind_weight_share"
]],["low_cost","high_cost","low_turnover","high_turnover","any_trade_cap_bind",
    "trade_cap_bind_weight_share","any_pos_cap_bind","pos_cap_bind_weight_share"])

sl["low_cost_gap"]=sl["low_cost_HMM"]-sl["low_cost_RF"]
sl["high_cost_gap"]=sl["high_cost_HMM"]-sl["high_cost_RF"]
sl["low_turn_gap"]=sl["low_turnover_HMM"]-sl["low_turnover_RF"]
sl["high_turn_gap"]=sl["high_turnover_HMM"]-sl["high_turnover_RF"]

sl_aum=sl.groupby("aum_millions",as_index=False).agg(
    low_cost_gap=("low_cost_gap","mean"),high_cost_gap=("high_cost_gap","mean"),
    low_turn_gap=("low_turn_gap","mean"),high_turn_gap=("high_turn_gap","mean"),
    hmm_trade_bind=("any_trade_cap_bind_HMM","mean"),rf_trade_bind=("any_trade_cap_bind_RF","mean"),
    hmm_trade_bind_share=("trade_cap_bind_weight_share_HMM","mean"),rf_trade_bind_share=("trade_cap_bind_weight_share_RF","mean"),
)
sl_aum["low_cost_gap_bps"]=1e4*sl_aum["low_cost_gap"]
sl_aum["high_cost_gap_bps"]=1e4*sl_aum["high_cost_gap"]
den=sl_aum["low_cost_gap"].abs()+sl_aum["high_cost_gap"].abs()
sl_aum["short_share_incremental_cost"]=np.where(den>0,sl_aum["low_cost_gap"].abs()/den,np.nan)

display(sl_aum.style.format({
    "aum_millions":"${:,.0f}m","low_cost_gap_bps":"{:+.2f}","high_cost_gap_bps":"{:+.2f}",
    "low_turn_gap":"{:+.3f}","high_turn_gap":"{:+.3f}","short_share_incremental_cost":"{:.1%}",
    "hmm_trade_bind":"{:.1%}","rf_trade_bind":"{:.1%}",
    "hmm_trade_bind_share":"{:.1%}","rf_trade_bind_share":"{:.1%}",
}))
sl_aum.to_csv(AUDIT_DIR/"short_sleeve_capacity_attribution.csv",index=False)

fig,ax=plt.subplots(figsize=(10,5))
d=sl_aum.sort_values("aum_millions"); x=np.arange(len(d)); w=.35
ax.bar(x-w/2,d["low_cost_gap_bps"],w,label="Low/short sleeve")
ax.bar(x+w/2,d["high_cost_gap_bps"],w,label="High/long sleeve")
ax.axhline(0,linewidth=1); ax.set_xticks(x)
ax.set_xticklabels([f"${v:,.0f}m" for v in d["aum_millions"]])
ax.set_ylabel("HMM − RF incremental cost (bps/rebalance)")
ax.set_title("Incremental HMM Cost by Sleeve"); ax.legend()
savefig(fig,"incremental_cost_by_sleeve.png")

In [ ]:
fig,ax=plt.subplots(figsize=(9,5))
d=sl_aum.sort_values("aum_millions")
ax.plot(d["aum_millions"],d["hmm_trade_bind"],marker="o",label="HMM")
ax.plot(d["aum_millions"],d["rf_trade_bind"],marker="o",label="RF")
ax.set_xscale("log"); ax.set_xlabel("AUM ($m, log scale)")
ax.set_ylabel("Share of dates with trade-cap binding")
ax.set_title("Trade-Cap Binding Frequency by AUM"); ax.legend()
savefig(fig,"trade_cap_binding_by_aum.png")

# Question 4 — HMM smoothing versus regime timing

This section uses three saved research artifacts:
- `hmm_weekly_state_summary.csv`
- `weekly_signal_persistence_turnover.csv`
- `hmm_weekly_cnn_rf_attribution_proxy.csv`

The central question is whether state/mapping variables explain **portfolio stability** even though earlier state tests did not explain incremental return.

In [ ]:
state_path=ANALYSIS_OUTPUT_DIR/"hmm_weekly_state_summary.csv"
persist_path=ANALYSIS_OUTPUT_DIR/"weekly_signal_persistence_turnover.csv"
attr_path=ANALYSIS_OUTPUT_DIR/"hmm_weekly_cnn_rf_attribution_proxy.csv"
for p in [state_path,persist_path,attr_path]:
    if not p.exists(): raise FileNotFoundError(f"Missing saved artifact: {p}")

state=pd.read_csv(state_path); persist=pd.read_csv(persist_path); attr=pd.read_csv(attr_path)
for d in [state,persist,attr]:
    d["Date"]=pd.to_datetime(d["Date"],errors="raise").dt.normalize()

hp=persist[persist["model"].astype(str).eq("HMM")].copy()
lp=persist[persist["model"].astype(str).eq("LogisticStack_WF")][["Date","total_target_turnover"]].copy()
lp=lp.rename(columns={"total_target_turnover":"logistic_turnover"})

smooth=hp.merge(state,on="Date",validate="one_to_one").merge(attr,on="Date",validate="one_to_one").sort_values("Date")
smooth["abs_change_cnn_influence"]=pd.to_numeric(smooth["cnn_influence_share"],errors="coerce").diff().abs()
smooth=smooth.merge(lp,on="Date",how="left",validate="one_to_one")
smooth["hmm_minus_logistic_turnover"]=smooth["total_target_turnover"]-smooth["logistic_turnover"]

regs=pd.concat([
    hac(smooth,"total_target_turnover",["state_entropy","max_state_prob","state_transition"]),
    hac(smooth,"total_target_turnover",["abs_change_cnn_influence"]),
    hac(smooth,"hmm_minus_logistic_turnover",["state_entropy","state_transition","abs_change_cnn_influence"]),
],ignore_index=True)

display(regs.style.format({"beta":"{:+.5f}","hac_t":"{:.3f}","p_value_two_sided":"{:.4f}","r_squared":"{:.4f}"}))
regs.to_csv(AUDIT_DIR/"state_smoothing_turnover_regressions.csv",index=False)

fig,ax=plt.subplots(figsize=(8,5))
d=smooth[["abs_change_cnn_influence","total_target_turnover"]].dropna()
ax.scatter(d["abs_change_cnn_influence"],d["total_target_turnover"],alpha=.45)
ax.set_xlabel("Absolute weekly change in CNN influence share")
ax.set_ylabel("HMM target turnover")
ax.set_title("Expert-Mapping Change and HMM Turnover")
savefig(fig,"mapping_change_vs_hmm_turnover.png")

In [ ]:
h100=portfolio_diag[(portfolio_diag["model"].astype(str)=="HMM") & np.isclose(portfolio_diag["aum_dollars"],PRIMARY_AUM)][
    ["date","total_turnover","low_replacement_frac","high_replacement_frac"]
].copy()
h100["mean_replacement_frac"]=(h100["low_replacement_frac"]+h100["high_replacement_frac"])/2
exec_state=h100.merge(state.rename(columns={"Date":"date"}),on="date",validate="one_to_one")

exec_regs=pd.concat([
    hac(exec_state,"mean_replacement_frac",["state_entropy","max_state_prob","state_transition"]),
    hac(exec_state,"total_turnover",["state_entropy","max_state_prob","state_transition"]),
],ignore_index=True)

display(exec_regs.style.format({"beta":"{:+.5f}","hac_t":"{:.3f}","p_value_two_sided":"{:.4f}","r_squared":"{:.4f}"}))
exec_regs.to_csv(AUDIT_DIR/"state_smoothing_executed_portfolio_100m.csv",index=False)

# Question 5 — Capacity-induced alpha dilution

The diagnostic tracks executed gross Sharpe, breadth, turnover and liquidity-constraint binding as AUM rises.

In [ ]:
cap=portfolio_diag.copy()
cap["mean_effective_n"]=(cap["low_effective_n"]+cap["high_effective_n"])/2
cap["mean_active_names"]=(cap["low_total_names"]+cap["high_total_names"])/2
cap["mean_sector_hhi"]=(cap["low_sector_hhi"]+cap["high_sector_hhi"])/2

cap=cap.groupby(["model","aum_dollars","aum_millions"],as_index=False).agg(
    mean_effective_n=("mean_effective_n","mean"),mean_active_names=("mean_active_names","mean"),
    mean_sector_hhi=("mean_sector_hhi","mean"),mean_turnover=("total_turnover","mean"),
    pos_cap_bind_pct=("any_pos_cap_bind","mean"),trade_cap_bind_pct=("any_trade_cap_bind","mean"),
    pos_cap_bind_share=("pos_cap_bind_weight_share","mean"),trade_cap_bind_share=("trade_cap_bind_weight_share","mean"),
)

perf=[]
for (m,a),g in accounting.groupby(["model","aum_dollars"]):
    gs=ann_stats(g["executed_gross_return"]); ns=ann_stats(g["net_return"])
    perf.append({"model":m,"aum_dollars":a,"executed_gross_sharpe":gs["sharpe"],"net_sharpe":ns["sharpe"],
                 "executed_gross_ann_vol":gs["ann_vol"],"net_ann_vol":ns["ann_vol"]})
cap=cap.merge(pd.DataFrame(perf),on=["model","aum_dollars"],validate="one_to_one")

display(cap.sort_values(["aum_dollars","model"]).style.format({
    "aum_millions":"${:,.0f}m","mean_effective_n":"{:.1f}","mean_active_names":"{:.1f}",
    "mean_sector_hhi":"{:.3f}","mean_turnover":"{:.3f}","pos_cap_bind_pct":"{:.1%}",
    "trade_cap_bind_pct":"{:.1%}","pos_cap_bind_share":"{:.1%}","trade_cap_bind_share":"{:.1%}",
    "executed_gross_sharpe":"{:.3f}","net_sharpe":"{:.3f}",
}))
cap.to_csv(AUDIT_DIR/"capacity_alpha_dilution_summary.csv",index=False)

In [ ]:
for y,label,title,name in [
    ("executed_gross_sharpe","Executed gross Sharpe","Executed Gross Signal Quality After Capacity Constraints","executed_gross_sharpe_by_aum.png"),
    ("mean_active_names","Mean active names per sleeve","Portfolio Breadth by AUM","active_names_by_aum.png"),
    ("mean_turnover","Realized portfolio turnover","Turnover Compression as AUM Increases","turnover_by_aum.png"),
]:
    fig,ax=plt.subplots(figsize=(9,5))
    for m,g in cap.groupby("model"):
        g=g.sort_values("aum_millions")
        ax.plot(g["aum_millions"],g[y],marker="o",label=m)
    ax.set_xscale("log"); ax.set_xlabel("AUM ($m, log scale)")
    ax.set_ylabel(label); ax.set_title(title); ax.legend()
    savefig(fig,name)

# Question 6 — Asymmetric hybrid diagnostic

The saved `portfolio_returns.csv` files are first validated to ensure

\[
R_{H-L}=R_{High}-R_{Low}.
\]

Only then are the diagnostic combinations formed. These are not jointly reoptimized portfolios.

In [ ]:
def sleeve_cols(df):
    cols=[str(c) for c in df.columns]
    hl=next((c for c in cols if c.lower().replace("_","-") in {"h-l","hl"}),None)
    if hl is None: raise RuntimeError(f"No H-L column in {cols}")
    lower={c.lower():c for c in cols}
    if "low" in lower and "high" in lower:
        return lower["low"],lower["high"],hl
    nums=[]
    for c in cols:
        try: nums.append((float(c),c))
        except: pass
    if len(nums)<2: raise RuntimeError(f"Cannot identify sleeve columns in {cols}")
    nums=sorted(nums)
    return nums[0][1],nums[-1][1],hl

hyb=[]
common=sorted(set(a for m,a in portfolio_returns if m=="RF") & set(a for m,a in portfolio_returns if m=="HMM"))
for a in common:
    rf=portfolio_returns[("RF",a)].copy(); hm=portfolio_returns[("HMM",a)].copy()
    rl,rh,rhl=sleeve_cols(rf); hl,hh,hhl=sleeve_cols(hm)

    rres=(pd.to_numeric(rf[rh])-pd.to_numeric(rf[rl])-pd.to_numeric(rf[rhl])).abs().max()
    hres=(pd.to_numeric(hm[hh])-pd.to_numeric(hm[hl])-pd.to_numeric(hm[hhl])).abs().max()
    if max(rres,hres)>1e-9:
        raise RuntimeError(f"Sleeve identity failed at ${a/1e6:.0f}m; do not construct hybrid.")

    d=pd.DataFrame({
        "RF":pd.to_numeric(rf[rhl]),
        "HMM":pd.to_numeric(hm[hhl]),
        "HMM_long_RF_short":pd.to_numeric(hm[hh])-pd.to_numeric(rf[rl]),
        "RF_long_HMM_short":pd.to_numeric(rf[rh])-pd.to_numeric(hm[hl]),
    }).dropna()

    for s in d:
        st=ann_stats(d[s])
        hyb.append({"aum_dollars":a,"aum_millions":a/1e6,"strategy":s,**st})

hyb=pd.DataFrame(hyb)
display(hyb.sort_values(["aum_dollars","strategy"]).style.format({
    "aum_millions":"${:,.0f}m","ann_mean":"{:.2%}","ann_vol":"{:.2%}","sharpe":"{:.3f}",
}))
hyb.to_csv(AUDIT_DIR/"diagnostic_asymmetric_hybrid_summary.csv",index=False)

fig,ax=plt.subplots(figsize=(10,5))
for s,g in hyb.groupby("strategy"):
    g=g.sort_values("aum_millions")
    ax.plot(g["aum_millions"],g["sharpe"],marker="o",label=s)
ax.set_xscale("log"); ax.set_xlabel("AUM ($m, log scale)")
ax.set_ylabel("Annualized Sharpe"); ax.set_title("Diagnostic Asymmetric Sleeve Combinations")
ax.legend()
savefig(fig,"diagnostic_hybrid_sharpe_by_aum.png")

In [ ]:
# =============================================================================
# FOLLOW-UP 1 — EXACT LONG/SHORT VARIANCE–COVARIANCE DECOMPOSITION
# =============================================================================
#
# Question:
# What actually produces the HMM–RF volatility difference?
#
# For H-L:
#
#     Var(H - L) = Var(H) + Var(L) - 2 Cov(H, L)
#
# The decomposition below is exact up to floating-point error.
# =============================================================================

variance_rows = []

common_aums = sorted(
    set(a for m, a in portfolio_returns if m == "RF")
    & set(a for m, a in portfolio_returns if m == "HMM")
)

for aum in common_aums:

    for model in ("RF", "HMM"):

        returns = portfolio_returns[
            (model, aum)
        ].copy()

        low_col, high_col, hl_col = sleeve_cols(
            returns
        )

        panel = pd.DataFrame(
            {
                "high": pd.to_numeric(
                    returns[high_col],
                    errors="coerce",
                ),
                "low": pd.to_numeric(
                    returns[low_col],
                    errors="coerce",
                ),
                "hl": pd.to_numeric(
                    returns[hl_col],
                    errors="coerce",
                ),
            }
        ).dropna()

        # Verify the saved accounting identity.
        identity_error = (
            panel["high"]
            - panel["low"]
            - panel["hl"]
        ).abs().max()

        if identity_error > 1e-9:
            raise RuntimeError(
                f"H-L identity failed for "
                f"{model} @ ${aum / 1e6:.0f}m: "
                f"{identity_error:.3e}"
            )

        var_high = panel["high"].var(
            ddof=1
        )
        var_low = panel["low"].var(
            ddof=1
        )
        cov_high_low = panel[
            ["high", "low"]
        ].cov(ddof=1).iloc[0, 1]

        covariance_term = (
            -2.0 * cov_high_low
        )

        decomposed_var = (
            var_high
            + var_low
            + covariance_term
        )

        direct_var = panel["hl"].var(
            ddof=1
        )

        reconstruction_error = (
            decomposed_var
            - direct_var
        )

        if abs(reconstruction_error) > 1e-12:
            raise RuntimeError(
                f"Variance decomposition failed for "
                f"{model} @ ${aum / 1e6:.0f}m: "
                f"{reconstruction_error:.3e}"
            )

        variance_rows.append(
            {
                "model": model,
                "aum_dollars": aum,
                "aum_millions": (
                    aum / 1e6
                ),
                "n_weeks": len(panel),

                # Annualized variance components.
                "high_variance": (
                    PERIODS_PER_YEAR
                    * var_high
                ),
                "low_variance": (
                    PERIODS_PER_YEAR
                    * var_low
                ),
                "covariance_term": (
                    PERIODS_PER_YEAR
                    * covariance_term
                ),
                "hl_variance": (
                    PERIODS_PER_YEAR
                    * direct_var
                ),

                # Easier-to-read volatility.
                "hl_ann_vol": np.sqrt(
                    PERIODS_PER_YEAR
                    * direct_var
                ),

                "high_low_correlation": (
                    panel["high"].corr(
                        panel["low"]
                    )
                ),
            }
        )


variance_decomp = pd.DataFrame(
    variance_rows
)


# -----------------------------------------------------------------------------
# RF minus HMM decomposition
#
# Positive total gap:
#     RF variance > HMM variance
#
# Positive component:
#     that component contributes to HMM's lower variance.
# -----------------------------------------------------------------------------

wide = (
    variance_decomp
    .pivot(
        index=[
            "aum_dollars",
            "aum_millions",
        ],
        columns="model",
        values=[
            "high_variance",
            "low_variance",
            "covariance_term",
            "hl_variance",
            "hl_ann_vol",
            "high_low_correlation",
        ],
    )
)

wide.columns = [
    f"{metric}_{model}"
    for metric, model
    in wide.columns
]

variance_gap = (
    wide
    .reset_index()
)

variance_gap[
    "rf_minus_hmm_high_variance"
] = (
    variance_gap[
        "high_variance_RF"
    ]
    - variance_gap[
        "high_variance_HMM"
    ]
)

variance_gap[
    "rf_minus_hmm_low_variance"
] = (
    variance_gap[
        "low_variance_RF"
    ]
    - variance_gap[
        "low_variance_HMM"
    ]
)

variance_gap[
    "rf_minus_hmm_covariance_term"
] = (
    variance_gap[
        "covariance_term_RF"
    ]
    - variance_gap[
        "covariance_term_HMM"
    ]
)

variance_gap[
    "rf_minus_hmm_total_variance"
] = (
    variance_gap[
        "hl_variance_RF"
    ]
    - variance_gap[
        "hl_variance_HMM"
    ]
)

variance_gap[
    "decomposition_check"
] = (
    variance_gap[
        "rf_minus_hmm_high_variance"
    ]
    + variance_gap[
        "rf_minus_hmm_low_variance"
    ]
    + variance_gap[
        "rf_minus_hmm_covariance_term"
    ]
    - variance_gap[
        "rf_minus_hmm_total_variance"
    ]
)

if (
    variance_gap[
        "decomposition_check"
    ].abs().max()
    > 1e-12
):
    raise RuntimeError(
        "RF-minus-HMM variance-gap "
        "decomposition failed."
    )


# Convert variance contributions into squared percentage-point
# units for easier reading.
for col in [
    "rf_minus_hmm_high_variance",
    "rf_minus_hmm_low_variance",
    "rf_minus_hmm_covariance_term",
    "rf_minus_hmm_total_variance",
]:
    variance_gap[
        col + "_pp2"
    ] = (
        10_000.0
        * variance_gap[col]
    )


display_cols = [
    "aum_millions",
    "hl_ann_vol_RF",
    "hl_ann_vol_HMM",
    "rf_minus_hmm_high_variance_pp2",
    "rf_minus_hmm_low_variance_pp2",
    "rf_minus_hmm_covariance_term_pp2",
    "rf_minus_hmm_total_variance_pp2",
    "high_low_correlation_RF",
    "high_low_correlation_HMM",
]

print(
    "RF − HMM VARIANCE DECOMPOSITION"
)
print(
    "Positive component values contribute "
    "to HMM having lower H-L variance."
)

display(
    variance_gap[
        display_cols
    ].style.format(
        {
            "aum_millions": "${:,.0f}m",
            "hl_ann_vol_RF": "{:.2%}",
            "hl_ann_vol_HMM": "{:.2%}",
            "rf_minus_hmm_high_variance_pp2": "{:+.2f}",
            "rf_minus_hmm_low_variance_pp2": "{:+.2f}",
            "rf_minus_hmm_covariance_term_pp2": "{:+.2f}",
            "rf_minus_hmm_total_variance_pp2": "{:+.2f}",
            "high_low_correlation_RF": "{:+.3f}",
            "high_low_correlation_HMM": "{:+.3f}",
        }
    )
)

variance_decomp.to_csv(
    AUDIT_DIR
    / "long_short_variance_covariance_decomposition.csv",
    index=False,
)

variance_gap.to_csv(
    AUDIT_DIR
    / "rf_hmm_variance_gap_decomposition.csv",
    index=False,
)


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(10, 5)
)

plot_df = variance_gap.sort_values(
    "aum_millions"
)

x = np.arange(
    len(plot_df)
)

ax.bar(
    x,
    plot_df[
        "rf_minus_hmm_high_variance_pp2"
    ],
    label="Long/high variance",
)

ax.bar(
    x,
    plot_df[
        "rf_minus_hmm_low_variance_pp2"
    ],
    bottom=plot_df[
        "rf_minus_hmm_high_variance_pp2"
    ],
    label="Short/low variance",
)

bottom_two = (
    plot_df[
        "rf_minus_hmm_high_variance_pp2"
    ]
    + plot_df[
        "rf_minus_hmm_low_variance_pp2"
    ]
)

ax.bar(
    x,
    plot_df[
        "rf_minus_hmm_covariance_term_pp2"
    ],
    bottom=bottom_two,
    label="Covariance term",
)

ax.axhline(
    0.0,
    linewidth=1,
)

ax.set_xticks(
    x
)

ax.set_xticklabels(
    [
        f"${v:,.0f}m"
        for v in plot_df[
            "aum_millions"
        ]
    ]
)

ax.set_ylabel(
    "RF − HMM annualized variance contribution "
    "(percentage-points²)"
)

ax.set_title(
    "What Explains HMM's Lower H-L Variance?"
)

ax.legend()

savefig(
    fig,
    "rf_hmm_variance_gap_decomposition.png",
)

In [ ]:
# =============================================================================
# FOLLOW-UP 2 — SLEEVE-SPECIFIC COST PER UNIT OF TURNOVER
# =============================================================================
#
# Question:
# Why does incremental HMM turnover cost substantially more on the short side?
#
# Cost intensity:
#
#     total sleeve cost / total sleeve turnover
#
# Multiplied by 10,000, this is bps of portfolio cost per 1.0 unit of
# realized sleeve turnover.
# =============================================================================

cost_intensity_rows = []

for (
    model,
    aum,
    aum_m,
), g in portfolio_diag.groupby(
    [
        "model",
        "aum_dollars",
        "aum_millions",
    ]
):

    if model not in (
        "RF",
        "HMM",
    ):
        continue

    for sleeve in (
        "low",
        "high",
    ):

        cost = pd.to_numeric(
            g[f"{sleeve}_cost"],
            errors="coerce",
        )

        turnover = pd.to_numeric(
            g[f"{sleeve}_turnover"],
            errors="coerce",
        )

        valid = (
            cost.notna()
            & turnover.notna()
        )

        cost = cost.loc[
            valid
        ]

        turnover = turnover.loc[
            valid
        ]

        total_turnover = float(
            turnover.sum()
        )

        total_cost = float(
            cost.sum()
        )

        if total_turnover <= 0:
            cost_per_turnover = np.nan
        else:
            cost_per_turnover = (
                total_cost
                / total_turnover
            )

        # These are descriptive liquidity characteristics
        # of the saved portfolio sleeve.
        avg_spread = pd.to_numeric(
            g[
                f"{sleeve}_avg_spread_bps"
            ],
            errors="coerce",
        ).mean()

        avg_adv = pd.to_numeric(
            g[
                f"{sleeve}_avg_adv_musd"
            ],
            errors="coerce",
        ).mean()

        avg_sigma = pd.to_numeric(
            g[
                f"{sleeve}_avg_sigma"
            ],
            errors="coerce",
        ).mean()

        cost_intensity_rows.append(
            {
                "model": model,
                "aum_dollars": (
                    float(aum)
                ),
                "aum_millions": (
                    float(aum_m)
                ),
                "sleeve": sleeve,
                "total_cost": (
                    total_cost
                ),
                "total_turnover": (
                    total_turnover
                ),
                "cost_per_unit_turnover": (
                    cost_per_turnover
                ),
                "cost_per_unit_turnover_bps": (
                    10_000.0
                    * cost_per_turnover
                ),
                "avg_spread_bps": (
                    avg_spread
                ),
                "avg_adv_musd": (
                    avg_adv
                ),
                "avg_sigma": (
                    avg_sigma
                ),
            }
        )


sleeve_cost_intensity = pd.DataFrame(
    cost_intensity_rows
)


# -----------------------------------------------------------------------------
# Low/short versus high/long cost intensity within each model
# -----------------------------------------------------------------------------

intensity_wide = (
    sleeve_cost_intensity
    .pivot(
        index=[
            "model",
            "aum_dollars",
            "aum_millions",
        ],
        columns="sleeve",
        values=[
            "cost_per_unit_turnover_bps",
            "avg_spread_bps",
            "avg_adv_musd",
            "avg_sigma",
        ],
    )
)

intensity_wide.columns = [
    f"{metric}_{sleeve}"
    for metric, sleeve
    in intensity_wide.columns
]

intensity_wide = (
    intensity_wide
    .reset_index()
)

intensity_wide[
    "short_to_long_cost_intensity_ratio"
] = (
    intensity_wide[
        "cost_per_unit_turnover_bps_low"
    ]
    / intensity_wide[
        "cost_per_unit_turnover_bps_high"
    ]
)

intensity_wide[
    "short_minus_long_cost_intensity_bps"
] = (
    intensity_wide[
        "cost_per_unit_turnover_bps_low"
    ]
    - intensity_wide[
        "cost_per_unit_turnover_bps_high"
    ]
)


print(
    "SLEEVE COST PER UNIT OF TURNOVER"
)

display(
    intensity_wide.sort_values(
        [
            "aum_dollars",
            "model",
        ]
    ).style.format(
        {
            "aum_millions": "${:,.0f}m",
            "cost_per_unit_turnover_bps_low": "{:.2f}",
            "cost_per_unit_turnover_bps_high": "{:.2f}",
            "short_to_long_cost_intensity_ratio": "{:.2f}x",
            "short_minus_long_cost_intensity_bps": "{:+.2f}",
            "avg_spread_bps_low": "{:.2f}",
            "avg_spread_bps_high": "{:.2f}",
            "avg_adv_musd_low": "${:,.1f}m",
            "avg_adv_musd_high": "${:,.1f}m",
            "avg_sigma_low": "{:.2%}",
            "avg_sigma_high": "{:.2%}",
        }
    )
)


# -----------------------------------------------------------------------------
# Focus specifically on $100m
# -----------------------------------------------------------------------------

print(
    "\n$100m IMPLEMENTATION DETAIL"
)

display(
    intensity_wide[
        np.isclose(
            intensity_wide[
                "aum_dollars"
            ],
            PRIMARY_AUM,
        )
    ].style.format(
        {
            "aum_millions": "${:,.0f}m",
            "cost_per_unit_turnover_bps_low": "{:.2f}",
            "cost_per_unit_turnover_bps_high": "{:.2f}",
            "short_to_long_cost_intensity_ratio": "{:.2f}x",
            "short_minus_long_cost_intensity_bps": "{:+.2f}",
            "avg_spread_bps_low": "{:.2f}",
            "avg_spread_bps_high": "{:.2f}",
            "avg_adv_musd_low": "${:,.1f}m",
            "avg_adv_musd_high": "${:,.1f}m",
            "avg_sigma_low": "{:.2%}",
            "avg_sigma_high": "{:.2%}",
        }
    )
)


sleeve_cost_intensity.to_csv(
    AUDIT_DIR
    / "sleeve_cost_per_unit_turnover.csv",
    index=False,
)

intensity_wide.to_csv(
    AUDIT_DIR
    / "sleeve_cost_intensity_comparison.csv",
    index=False,
)


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(10, 5)
)

for (
    model,
    sleeve,
), g in sleeve_cost_intensity.groupby(
    [
        "model",
        "sleeve",
    ]
):

    g = g.sort_values(
        "aum_millions"
    )

    label = (
        f"{model} "
        f"{'short/low' if sleeve == 'low' else 'long/high'}"
    )

    ax.plot(
        g[
            "aum_millions"
        ],
        g[
            "cost_per_unit_turnover_bps"
        ],
        marker="o",
        label=label,
    )

ax.set_xscale(
    "log"
)

ax.set_xlabel(
    "AUM ($m, log scale)"
)

ax.set_ylabel(
    "Cost per 1.0 unit of sleeve turnover (bps)"
)

ax.set_title(
    "Sleeve-Specific Trading Cost Intensity"
)

ax.legend()

savefig(
    fig,
    "sleeve_cost_per_unit_turnover_by_aum.png",
)

In [ ]:
# =============================================================================
# FOLLOW-UP 3 — PAIRED CIRCULAR-BLOCK BOOTSTRAP FOR HYBRID SLEEVES
# =============================================================================
#
# Purpose
# -------
# Determine whether the diagnostic hybrid differences are statistically
# meaningful rather than small arithmetic differences.
#
# This follows the research inference convention:
#
#     5,000 bootstrap replications
#     8-week circular blocks
#     seed = 1729
#
# The same block indexes are applied to every strategy in a comparison,
# preserving same-week cross-strategy dependence.
#
# IMPORTANT:
# These hybrids remain diagnostic combinations of saved sleeve streams.
# They are NOT jointly reoptimized portfolios.
# =============================================================================

HYBRID_BOOT_REPS = 5_000
HYBRID_BLOCK_WEEKS = 8
HYBRID_BOOT_SEED = 1729

HYBRID_BOOT_AUMS = (
    25_000_000.0,
    100_000_000.0,
    1_000_000_000.0,
)


def _ann_sharpe_array(x):
    x = np.asarray(
        x,
        dtype=float,
    )

    mean = np.mean(
        x
    )

    vol = np.std(
        x,
        ddof=1,
    )

    if (
        not np.isfinite(vol)
        or vol <= 0
    ):
        return np.nan

    return float(
        mean
        / vol
        * np.sqrt(
            PERIODS_PER_YEAR
        )
    )


def _circular_block_indices(
    n,
    block_len,
    rng,
):
    """
    Same circular moving-block construction used by the
    project's statistical-validation utilities.
    """

    n = int(
        n
    )

    block_len = int(
        block_len
    )

    n_blocks = int(
        np.ceil(
            n
            / block_len
        )
    )

    starts = rng.randint(
        0,
        n,
        size=n_blocks,
    )

    offsets = np.arange(
        block_len,
        dtype=int,
    )

    pieces = []

    for start in starts:
        pieces.extend(
            (
                (
                    int(start)
                    + offsets
                )
                % n
            ).tolist()
        )

    return np.asarray(
        pieces[:n],
        dtype=int,
    )


def _build_hybrid_panel(
    aum,
):
    rf = portfolio_returns[
        ("RF", aum)
    ].copy()

    hmm = portfolio_returns[
        ("HMM", aum)
    ].copy()

    rf_low, rf_high, rf_hl = (
        sleeve_cols(
            rf
        )
    )

    hmm_low, hmm_high, hmm_hl = (
        sleeve_cols(
            hmm
        )
    )

    panel = pd.DataFrame(
        {
            "RF": pd.to_numeric(
                rf[rf_hl],
                errors="coerce",
            ),
            "HMM": pd.to_numeric(
                hmm[hmm_hl],
                errors="coerce",
            ),

            "HMM_long_RF_short": (
                pd.to_numeric(
                    hmm[hmm_high],
                    errors="coerce",
                )
                - pd.to_numeric(
                    rf[rf_low],
                    errors="coerce",
                )
            ),

            "RF_long_HMM_short": (
                pd.to_numeric(
                    rf[rf_high],
                    errors="coerce",
                )
                - pd.to_numeric(
                    hmm[hmm_low],
                    errors="coerce",
                )
            ),
        }
    )

    if panel.isna().any().any():
        raise RuntimeError(
            f"Hybrid panel contains missing values "
            f"at ${aum / 1e6:.0f}m."
        )

    return panel


comparisons = [
    (
        "HMM_long_RF_short",
        "RF",
    ),
    (
        "HMM_long_RF_short",
        "HMM",
    ),
    (
        "RF_long_HMM_short",
        "RF",
    ),
    (
        "RF_long_HMM_short",
        "HMM",
    ),
]


bootstrap_rows = []

for aum_i, aum in enumerate(
    HYBRID_BOOT_AUMS
):

    print(
        f"Bootstrapping hybrids @ "
        f"${aum / 1e6:,.0f}m..."
    )

    panel = _build_hybrid_panel(
        aum
    )

    n = len(
        panel
    )

    arrays = {
        col: panel[
            col
        ].to_numpy(
            dtype=float
        )
        for col in panel.columns
    }

    observed_sharpes = {
        col: _ann_sharpe_array(
            values
        )
        for col, values
        in arrays.items()
    }

    observed_ann_returns = {
        col: float(
            np.mean(values)
            * PERIODS_PER_YEAR
        )
        for col, values
        in arrays.items()
    }

    # One paired bootstrap draw is used for all four
    # strategy streams within the AUM.
    rng = np.random.RandomState(
        HYBRID_BOOT_SEED
        + 1000 * aum_i
    )

    sharpe_draws = {
        col: np.empty(
            HYBRID_BOOT_REPS,
            dtype=float,
        )
        for col
        in arrays
    }

    ann_return_draws = {
        col: np.empty(
            HYBRID_BOOT_REPS,
            dtype=float,
        )
        for col
        in arrays
    }

    for b in range(
        HYBRID_BOOT_REPS
    ):

        idx = _circular_block_indices(
            n=n,
            block_len=HYBRID_BLOCK_WEEKS,
            rng=rng,
        )

        for col, values in (
            arrays.items()
        ):

            draw = values[
                idx
            ]

            sharpe_draws[
                col
            ][b] = (
                _ann_sharpe_array(
                    draw
                )
            )

            ann_return_draws[
                col
            ][b] = (
                np.mean(
                    draw
                )
                * PERIODS_PER_YEAR
            )

    for strategy_a, strategy_b in (
        comparisons
    ):

        sr_diff = (
            sharpe_draws[
                strategy_a
            ]
            - sharpe_draws[
                strategy_b
            ]
        )

        ret_diff = (
            ann_return_draws[
                strategy_a
            ]
            - ann_return_draws[
                strategy_b
            ]
        )

        finite_sr = (
            np.isfinite(
                sr_diff
            )
        )

        sr_diff = sr_diff[
            finite_sr
        ]

        finite_ret = (
            np.isfinite(
                ret_diff
            )
        )

        ret_diff = ret_diff[
            finite_ret
        ]

        sr_ci_low, sr_ci_high = (
            np.quantile(
                sr_diff,
                [
                    0.025,
                    0.975,
                ],
            )
        )

        ret_ci_low, ret_ci_high = (
            np.quantile(
                ret_diff,
                [
                    0.025,
                    0.975,
                ],
            )
        )

        bootstrap_rows.append(
            {
                "aum_dollars": aum,
                "aum_millions": (
                    aum / 1e6
                ),
                "strategy_a": (
                    strategy_a
                ),
                "strategy_b": (
                    strategy_b
                ),
                "n_weeks": n,
                "block_weeks": (
                    HYBRID_BLOCK_WEEKS
                ),
                "bootstrap_reps": (
                    len(
                        sr_diff
                    )
                ),

                "sharpe_a": (
                    observed_sharpes[
                        strategy_a
                    ]
                ),
                "sharpe_b": (
                    observed_sharpes[
                        strategy_b
                    ]
                ),
                "observed_delta_sharpe": (
                    observed_sharpes[
                        strategy_a
                    ]
                    - observed_sharpes[
                        strategy_b
                    ]
                ),
                "delta_sharpe_ci_low": (
                    sr_ci_low
                ),
                "delta_sharpe_ci_high": (
                    sr_ci_high
                ),
                "bootstrap_prob_delta_sharpe_gt_0": (
                    np.mean(
                        sr_diff > 0.0
                    )
                ),

                "ann_return_a": (
                    observed_ann_returns[
                        strategy_a
                    ]
                ),
                "ann_return_b": (
                    observed_ann_returns[
                        strategy_b
                    ]
                ),
                "observed_delta_ann_return": (
                    observed_ann_returns[
                        strategy_a
                    ]
                    - observed_ann_returns[
                        strategy_b
                    ]
                ),
                "delta_ann_return_ci_low": (
                    ret_ci_low
                ),
                "delta_ann_return_ci_high": (
                    ret_ci_high
                ),
                "bootstrap_prob_delta_ann_return_gt_0": (
                    np.mean(
                        ret_diff > 0.0
                    )
                ),
            }
        )


hybrid_bootstrap = pd.DataFrame(
    bootstrap_rows
)


print(
    "\nPAIRED BLOCK-BOOTSTRAP "
    "HYBRID INFERENCE"
)

display(
    hybrid_bootstrap.style.format(
        {
            "aum_millions": "${:,.0f}m",
            "sharpe_a": "{:.3f}",
            "sharpe_b": "{:.3f}",
            "observed_delta_sharpe": "{:+.3f}",
            "delta_sharpe_ci_low": "{:+.3f}",
            "delta_sharpe_ci_high": "{:+.3f}",
            "bootstrap_prob_delta_sharpe_gt_0": "{:.3f}",
            "ann_return_a": "{:.2%}",
            "ann_return_b": "{:.2%}",
            "observed_delta_ann_return": "{:+.2%}",
            "delta_ann_return_ci_low": "{:+.2%}",
            "delta_ann_return_ci_high": "{:+.2%}",
            "bootstrap_prob_delta_ann_return_gt_0": "{:.3f}",
        }
    )
)


hybrid_bootstrap.to_csv(
    AUDIT_DIR
    / "diagnostic_hybrid_paired_block_bootstrap.csv",
    index=False,
)

print(
    "\nSaved:",
    AUDIT_DIR
    / "diagnostic_hybrid_paired_block_bootstrap.csv",
)

# Final interpretation checklist


1. **Sector mechanism:** Is lower HMM HHI persistent, and does the HHI gap align with the realized-volatility gap?
2. **Cost mechanism:** What fraction of the HMM–RF cost gap is turnover quantity versus cost intensity? Is the direct gap mostly market impact?
3. **Short sleeve:** What share of incremental HMM cost and turnover originates in the short sleeve?
4. **Smoothing:** Do state variables predict turnover/replacement? Do changes in CNN influence explain churn more clearly?
5. **Capacity dilution:** Does executed gross Sharpe fall as binding rises and turnover compresses? Does breadth increase?
6. **Hybrid:** Does HMM-long/RF-short dominate the opposite asymmetric combination strongly enough to justify a future jointly reoptimized test?

All generated tables and figures are saved under `FUSION_ANALYSIS_DIR/economic_mechanism_audit/`.